In [1]:
#Importing Libraries

from collections import Counter
import json
from pathlib import Path
import random
import pandas as pd
import spacy

In [2]:
pip show spacy

Name: spacy
Version: 3.8.14
Summary: Industrial-strength Natural Language Processing (NLP) in Python
Home-page: https://spacy.io
Author: Explosion
Author-email: contact@explosion.ai
License: MIT
Location: C:\Users\aslam\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: catalogue, confection, cymem, jinja2, murmurhash, numpy, packaging, preshed, pydantic, requests, setuptools, spacy-legacy, spacy-loggers, srsly, thinc, tqdm, typer, wasabi, weasel
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
PROJECT_ROOT = BASE_DIR / "Data" / "Processed"
SPLIT_DATA = BASE_DIR / "Data" / "Splits"
SRC = BASE_DIR / "Src" / "Training"
print(BASE_DIR)
print(RAW_DIR)
print(PROJECT_ROOT)
print(SPLIT_DATA)
print(SRC)

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Splits
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Src\Training


In [4]:
# With spaCy, convert raw text + character offsets into proper Span objects.

import spacy
import json

nlp = spacy.blank("en")

with open(RAW_DIR/'synthetic_notes.json', "r", encoding="utf-8") as file:
    data = json.load(file)

# Pick one record
sample = data[0]

# Extract the raw text
text = sample["text"]

# Tokenize the text
doc = nlp.make_doc(text)

# Prepare an empty list for entity spans
ents = []

# oop through each entity annotation
for ent in sample["entities"]:

    # Convert character offsets → spaCy Span
    span = doc.char_span(
        ent["start"],
        ent["end"],
        label=ent["label"]
    )
    # Append valid spans
    if span is not None:
        ents.append(span)
        
# Assign the spans to the Doc
doc.ents = ents

# Verify

print(text)
print('\n')

for ent in doc.ents:
    print(ent.text, ':', ent.label_)

print('\n')

for ent in sample["entities"]:
    
    print(
        sample["text"][ent["start"]:ent["end"]],
        "->",
        ent["label"]
    )

Known case of Influenza. Advised to take Budesonide 250 mg once daily.


Influenza : DISEASE
Budesonide : MEDICATION
250 mg once daily : DOSAGE


Influenza -> DISEASE
Budesonide -> MEDICATION
250 mg once daily -> DOSAGE


In [5]:
print(
    "Expected:", len(sample["entities"]),
    "Created:", len(doc.ents)
)

Expected: 3 Created: 3


In [6]:
# Validate Entire Dataset

def count_alignment_errors(data, nlp):
    # Initialize counters
    total_entities = 0
    failed_entities = 0

    # Loop through every record
    for record in data:

        # Tokenize the text
        doc = nlp.make_doc(record["text"])

        # Loop through each entity in the record
        for ent in record["entities"]:

            # Count the entity
            total_entities += 1

            # Try to convert offsets → spaCy Span
            span = doc.char_span(
                ent["start"],
                ent["end"],
                label=ent["label"]
            )

            if span is None:
                failed_entities += 1

    print("Total Entities :", total_entities)
    print("Failed Entities:", failed_entities)

    return failed_entities

#ChecK
count_alignment_errors(data, nlp)

Total Entities : 16924
Failed Entities: 3


3

In [7]:
# Generate train.spacy / valid.spacy / test.spacy

# Import the binary container class.
from spacy.tokens import DocBin

def create_docbin(data, output_path, nlp):

    # Create a DocBin
    doc_bin = DocBin()

    # Loop through each record
    for record in data:

        # Tokenize the text
        doc = nlp.make_doc(record["text"])

        # Build entity spans
        ents = []

        # Loop through entities
        for ent in record["entities"]:

            # Convert offsets → spans
            span = doc.char_span(
                ent["start"],
                ent["end"],
                label=ent["label"]
            )

            if span is not None:
                ents.append(span)

        # Assign entities to the Doc
        doc.ents = ents
        # Add the Doc to the DocBin
        doc_bin.add(doc)

    # Save to disk
    doc_bin.to_disk(output_path)

    print(f"Saved: {output_path}")



In [8]:
from spacy.util import filter_spans

def create_spacy_doc(record, nlp):

    doc = nlp.make_doc(record["text"])

    ents = []

    for ent in record["entities"]:

        span = doc.char_span(
            ent["start"],
            ent["end"],
            label=ent["label"]
        )

        if span is not None:
            ents.append(span)

    doc.ents = filter_spans(ents)

    return doc

In [14]:
from spacy.tokens import DocBin

def create_spacy_dataset(data, output_path, nlp):
    doc_bin = DocBin()

    for record in data:
        doc = create_spacy_doc(record, nlp)
        doc_bin.add(doc)

    doc_bin.to_disk(output_path)
print(f"Saved: {output_path}")


Saved: D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed\test.spacy


In [10]:
# Load split records

import json

with open(SPLIT_DATA / "train_records.json", "r", encoding="utf-8") as f:
    train_records = json.load(f)

with open(SPLIT_DATA / "val_records.json", "r", encoding="utf-8") as f:
    val_records = json.load(f)

with open(SPLIT_DATA / "test_records.json", "r", encoding="utf-8") as f:
    test_records = json.load(f)


In [11]:
# Save spacy split records/files

output_path = PROJECT_ROOT/'train.spacy'
create_spacy_dataset(train_records, output_path, nlp)

output_path = PROJECT_ROOT/'valid.spacy'
create_spacy_dataset(val_records, output_path, nlp)

output_path = PROJECT_ROOT/'test.spacy'
create_spacy_dataset(test_records, output_path, nlp)

Saved: D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed\train.spacy
Saved: D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed\valid.spacy
Saved: D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed\test.spacy


In [12]:
# verify

from pathlib import Path

for file in ["train.spacy", "valid.spacy", "test.spacy"]:
    p = PROJECT_ROOT / file
    print(file, p.exists(), p.stat().st_size if p.exists() else "Missing")

train.spacy True 349987
valid.spacy True 47471
test.spacy True 47664
